# Classical LDLC decoding

Construct a small LDLC, decode one noisy lattice word, and estimate its symbol-error rate. The defaults are deliberately small; increase `LATTICEDECODER_EXAMPLE_SAMPLES` for longer runs.

In [ ]:
import Pkg
repo_root = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : normpath(joinpath(pwd(), ".."))
Pkg.activate(repo_root)

using LinearAlgebra
using LatticeDecoder
using Random

samples_per_point = parse(Int, get(ENV, "LATTICEDECODER_EXAMPLE_SAMPLES", "20"))

In [ ]:
Random.seed!(2026)
H = classical_ldlc(3, 12, true)
G = generator_matrix(H)
problem = ClassicalDecodingProblem(H, G)

sigma = 0.12
decoder = LDLCDecoder(
    initialize_tanner_graph(H);
    schedule=:serial,
    algorithm=:lsd,
    sigma,
    max_iterations=8,
)

rng = MersenneTwister(10)
symbols = rand(rng, 0:1, size(H, 2))
received = encode(symbols, G) + sample_error(rng, sigma, size(H, 2))
soft_estimate = run_decoder!(decoder, received)
decoded_symbols = hard_decision(soft_estimate, H)

(
    transmitted=symbols,
    decoded=decoded_symbols,
    symbol_errors=count_symbol_errors(decoded_symbols, symbols),
)

In [ ]:
sigmas = [0.08, 0.12, 0.16]
estimates = [
    estimate_symbol_error_rate!(
        MersenneTwister(100),
        LDLCDecoder(
            initialize_tanner_graph(H);
            schedule=:serial,
            algorithm=:lsd,
            sigma,
            max_iterations=8,
        ),
        problem;
        samples=samples_per_point,
    )
    for sigma in sigmas
]

[
    (
        sigma=sigma,
        errors=result.events,
        trials=result.trials,
        rate=result.rate,
        interval=(result.lower, result.upper),
    )
    for (sigma, result) in zip(sigmas, estimates)
]

For publication runs, record the package commit, Julia version, seed, decoder configuration, raw event count, and total number of symbol trials.